In [1]:
import math
import torch
import torch.nn as nn
import tiktoken

# ---------- Config (must match training) ----------
GPT_CONFIG_163M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

# ---------- Model ----------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, _ = x.shape
        keys = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        context_vec = torch.nn.functional.scaled_dot_product_attention(
            queries, keys, values,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=True,
        ).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        return torch.nn.functional.layer_norm(
            x, self.scale.shape, self.scale, self.shift, self.eps
        )


class GELU(nn.Module):
    def forward(self, x):
        return torch.nn.functional.gelu(x, approximate="tanh")


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.drop_shortcut(self.att(self.norm1(x))) + shortcut
        shortcut = x
        x = self.drop_shortcut(self.ff(self.norm2(x))) + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        _, seq_len = in_idx.shape
        x = self.tok_emb(in_idx) + self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)


# ---------- Tokenizer helpers ----------
tokenizer = tiktoken.get_encoding("gpt2")

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(encoded).unsqueeze(0)

def format_input(entry):
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry.get("input") else ""
    return instruction_text + input_text

# # ---------- Load fine-tuned weights ----------
# CKPT = "my-finetuned-gpt.pth"
# chat_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# chat_model = GPTModel(GPT_CONFIG_163M)
# chat_model.load_state_dict(torch.load(CKPT, map_location=chat_device, weights_only=True))
# chat_model.to(chat_device).eval()
# print("Loaded:", CKPT, "on", chat_device)

# # ---------- Reply function (used by the Gradio cell) ----------
# def generate_reply(text, max_new_tokens=200, temperature=0.7, top_k=40, rep_penalty=1.2):
#     prompt = format_input({"instruction": text, "input": ""}) + "\n\n### Response:\n"
#     ids = text_to_token_ids(prompt, tokenizer).to(chat_device)
#     start = ids.shape[1]
#     ctx = chat_model.pos_emb.weight.shape[0]

#     with torch.no_grad():
#         for _ in range(max_new_tokens):
#             logits = chat_model(ids[:, -ctx:])[:, -1, :].float()

#             # discourage repeating tokens already generated
#             for t in set(ids[0, start:].tolist()):
#                 v = logits[0, t]
#                 logits[0, t] = v / rep_penalty if v > 0 else v * rep_penalty

#             # top-k + temperature sampling
#             top_vals, _ = torch.topk(logits, top_k)
#             logits = torch.where(
#                 logits < top_vals[:, -1:], torch.full_like(logits, float("-inf")), logits
#             )
#             probs = torch.softmax(logits / temperature, dim=-1)
#             nxt = torch.multinomial(probs, num_samples=1)

#             if nxt.item() == 50256:  # <|endoftext|>
#                 break
#             ids = torch.cat((ids, nxt), dim=1)

#     reply = tokenizer.decode(ids[0, start:].tolist())
#     return reply.split("### Instruction")[0].strip()

# # quick sanity check
# print(generate_reply("What is the capital of France?", max_new_tokens=40))

Fine-tuning to follow
instructions

Preparing a dataset for supervised instruction 
fine-tuning

In [2]:

import pyarrow.parquet as pq
from pathlib import Path

FT_DATA_DIR = Path(
    r"D:\built a llm from scratch 124million parameters\fineweb-edu\fine tuning"
)

parquet_files = sorted(FT_DATA_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"No Parquet files found in {FT_DATA_DIR}"
    )

data = []

for file_path in parquet_files:
    print("Loading:", file_path.name)

    parquet_file = pq.ParquetFile(file_path)

    for batch in parquet_file.iter_batches(
        batch_size=1000,
        columns=["messages", "source"]
    ):
        for row in batch.to_pylist():
            messages = row.get("messages") or []
            source = row.get("source", "")

            pending_user = None
            assistant_parts = []

            for message in messages:
                role = str(message.get("role", "")).lower()
                content = message.get("content") or ""

                if role in ("user", "human"):
                    if pending_user is not None and assistant_parts:
                        data.append({
                            "instruction": pending_user,
                            "input": "",
                            "output": "\n\n".join(assistant_parts),
                            "source": source
                        })

                    pending_user = content
                    assistant_parts = []

                elif role in ("assistant", "gpt"):
                    if pending_user is not None:
                        assistant_parts.append(content)

            if pending_user is not None and assistant_parts:
                data.append({
                    "instruction": pending_user,
                    "input": "",
                    "output": "\n\n".join(assistant_parts),
                    "source": source
                })

print("Total instruction examples:", len(data))
print("\nExample entry:")
print(data[0] if data else "No usable examples found")


Loading: train-00000-of-00004.parquet
Loading: train-00001-of-00004.parquet
Loading: train-00002-of-00004.parquet
Loading: train-00003-of-00004.parquet
Total instruction examples: 1031866

Example entry:
{'instruction': 'I need you to edit something for me. This is the text I wrote, \n\n"Me and my friends have been waiting for a long time to go back to the movies and catch a movie we been waiting on. Last nite I finely went to go buy tickets and when I got to the movie theater the tickets were sold out, so I was pretty pist. We were all pist. So today we deside if we wanted to go to a game instead, but I dont think we will go to a game now."', 'input': '', 'output': 'Here\'s a revised version of your text with some suggested edits to improve grammar, clarity, and overall flow:\n\n"My friends and I have been waiting a long time to return to the movies and see a film we\'ve been eagerly anticipating. Last night, I finally went to buy tickets, but when I arrived at the theater, they were 

In [3]:
print("Example entry:\n", data[50])

Example entry:
 {'instruction': 'How can I efficiently run and debug the NodeJS code for queries and UI components separately, while keeping them in the same hierarchy and running them independently? The current code is written in TypeScript and handles running queries against databases and caching results in SQLite. The UI component will be built using electron, solidjs, and vite for hot-reloading. Provide a detailed set of steps that involve multiple-step reasoning to help me understand how to call these functions, write queries, execute them, and debug the code using breakpoints, etc. in both the "js_query" and "js_query_ui" folders.', 'input': '', 'output': 'To efficiently run and debug NodeJS code for queries and UI components separately, while keeping them in the same hierarchy and running them independently, you can follow these steps:\n\n1. Create two folders, one for the queries and one for the UI component. Let\'s name these folders "js_query" and "js_query_ui", respectively.

Implementing the prompt formatting function

In [4]:
def format_input(entry):
    instruction_text = (
f"Below is an instruction that describes a task. "
f"Write a response that appropriately completes the request."
f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text + input_text

In [5]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
How can I efficiently run and debug the NodeJS code for queries and UI components separately, while keeping them in the same hierarchy and running them independently? The current code is written in TypeScript and handles running queries against databases and caching results in SQLite. The UI component will be built using electron, solidjs, and vite for hot-reloading. Provide a detailed set of steps that involve multiple-step reasoning to help me understand how to call these functions, write queries, execute them, and debug the code using breakpoints, etc. in both the "js_query" and "js_query_ui" folders.

### Response:
To efficiently run and debug NodeJS code for queries and UI components separately, while keeping them in the same hierarchy and running them independently, you can follow these steps:

1. Create two folders, one for the queries and one for the UI co

Before we move on to setting up the PyTorch data loaders in the next section, let’s
divide the dataset into training, validation, and test sets analogous

In [6]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_portion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

import random
random.Random(123).shuffle(train_data)   # NEW: shuffle before taking a subset
N_SFT = 100_000                          # NEW: examples per epoch (None = all ~877k, takes days)
if N_SFT:
    train_data = train_data[:N_SFT]
val_data = val_data[:5000]               # NEW: smaller val set = faster eval

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 100000
Validation set length: 5000
Test set length: 103186



Organizing data into training batches

Implementing an instruction dataset class

In [7]:

import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __getitem__(self, index):
        entry = self.data[index]
        prompt_ids = self.tokenizer.encode(
            format_input(entry) + "\n\n### Response:\n", disallowed_special=()
        )
        resp_ids = self.tokenizer.encode(entry["output"], disallowed_special=())
        return prompt_ids + resp_ids, len(prompt_ids)   # CHANGED: also return prompt length

    def __len__(self):
        return len(self.data)


2.5 Replace
padding tokens
with placeholders

In [8]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item) + 1 for item, _ in batch)   # CHANGED
    inputs_lst, targets_lst = [], []

    for item, prompt_len in batch:                               # CHANGED
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )

        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()

        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        targets[:prompt_len - 1] = ignore_index                  # NEW: loss on response only

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [10]:
from functools import partial
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

In [11]:

from torch.utils.data import DataLoader
from functools import partial

num_workers = 0
batch_size = 4   # CHANGED (was 1)

torch.manual_seed(123)

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=512   # CHANGED (was 1024)
)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Test examples:", len(test_dataset))

inputs, targets = next(iter(train_loader))

print("Input shape:", inputs.shape)
print("Target shape:", targets.shape)


Training examples: 100000
Validation examples: 5000
Test examples: 103186
Input shape: torch.Size([4, 512])
Target shape: torch.Size([4, 512])


In [12]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GPTModel(GPT_CONFIG_163M)
checkpoint = torch.load("pretrained_gpt124m_before_sft.pth", map_location=device, weights_only=True)  # CHANGED
model.load_state_dict(checkpoint)
model.to(device)
model.train()

print("Loaded my own pretrained model!")

Loaded my own pretrained model!


In [13]:
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(input_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Let's consider a dataset consisting of exam scores from 500 students across five different high schools. The dataset includes variables for the student's score, school name, and student ID. The average score is 78, the median score is 80, and the standard deviation is 8.

Given the information provided, what can we infer about the distribution of the exam scores across the 500 students?


In [14]:

def generate(
    model,
    idx,
    max_new_tokens,
    context_size,
    temperature=0.0,
    top_k=None,
    eos_id=None
):
    # Move input tokens to the same device as the model
    device = next(model.parameters()).device
    idx = idx.to(device)

    for _ in range(max_new_tokens):

        # Keep only the tokens that fit in the context window
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        # Get predictions for the next token
        logits = logits[:, -1, :]

        # Keep only the top-k most likely tokens
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1].unsqueeze(-1)

            logits = torch.where(
                logits < min_val,
                torch.full_like(logits, float("-inf")),
                logits
            )

        # Choose the next token
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )
        else:
            idx_next = torch.argmax(
                logits,
                dim=-1,
                keepdim=True
            )

        # Stop if the end-of-sequence token is generated
        if eos_id is not None and idx_next.item() == eos_id:
            break

        # Append the new token to the sequence
        idx = torch.cat((idx, idx_next), dim=1)

    return idx


In [15]:
BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

Baseline Evaluation Before Instruction Fine-Tuning

In [16]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.float().flatten(0, 1), target_batch.flatten()
    )
    return loss

In [17]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.

    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            total_loss += loss.item()
        else:
            break

    return total_loss / num_batches

In [18]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
            val_loader, model, device, num_batches=eval_iter
        )
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()

In [19]:
import math

def get_lr(update, warmup_updates, total_updates, peak_lr, min_lr):
    if update < warmup_updates:
        return peak_lr * (update + 1) / warmup_updates
    if update >= total_updates:
        return min_lr
    progress = (update - warmup_updates) / (total_updates - warmup_updates)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

In [20]:
model = GPTModel(GPT_CONFIG_163M)
model.load_state_dict(torch.load("pretrained_gpt124m_before_sft.pth",
                                 map_location="cpu", weights_only=True))
model.to(device).eval()
print("Loaded pre-SFT checkpoint")

Loaded pre-SFT checkpoint


In [21]:
# Evaluate generation before supervised fine-tuning
model.eval()

input_text = (
    format_input(val_data[0])
    + "\n\n### Response:\n"
)

input_ids = text_to_token_ids(input_text, tokenizer).to(device)

token_ids = generate(
    model=model,
    idx=input_ids,
    max_new_tokens=100,
    context_size=GPT_CONFIG_163M["context_length"],
    eos_id=50256,
)

# Decode only the newly generated tokens
response_ids = token_ids[0, input_ids.shape[1]:]
response_text = tokenizer.decode(response_ids.tolist()).strip()

print("Pre-SFT model response:")
print(response_text)

Pre-SFT model response:
The average score for the student is 79, the median score is 80, and the standard deviation is 8.
### Response:
The average score for the student is 79, the median score is 80, and the standard deviation is 8.
### Response:
The average score for the student is 79, the median score is 80, and the standard deviation is 8.
### Response:
The average score for the student is 79, the median score is 80, and the standard deviation


In [22]:
model.to(device)
torch.manual_seed(123)
with torch.no_grad():
    train_loss = calc_loss_loader(
train_loader, model, device, num_batches=5
    )
    val_loss = calc_loss_loader(
val_loader, model, device, num_batches=5
)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 2.535192632675171
Validation loss: 2.813507652282715


In [23]:
print("Device:", device)
print("Model device:", next(model.parameters()).device)
print("Batches per epoch:", len(train_loader))
print("Batch size:", train_loader.batch_size)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
Model device: cuda:0
Batches per epoch: 25000
Batch size: 4
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


SFT v2: restarted from the pretrained checkpoint, 3 epochs on a shuffled subset, effective batch 32, warmup + cosine LR (peak 2e-5), loss on response tokens only, max length 512. Val loss is response-only now, so it is not comparable with the v1 numbers.

In [24]:
import time
from tqdm.auto import tqdm

EPOCHS = 3
ACCUM = 8                                  # 4 x 8 = effective batch of 32
PEAK_LR, MIN_LR, WARMUP = 2e-5, 2e-6, 100  # warmup + cosine decay
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# always start from the pretrained (pre-SFT) weights
model = GPTModel(GPT_CONFIG_163M)
model.load_state_dict(torch.load("pretrained_gpt124m_before_sft.pth", map_location="cpu", weights_only=True))
model.to(device).train()

optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=0.1, fused=True)

total_updates = EPOCHS * (len(train_loader) // ACCUM)
update, t0 = 0, time.time()

for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}")
    for micro, (inp, tgt) in enumerate(pbar, 1):
        loss = calc_loss_batch(inp, tgt, model, device)
        if torch.isfinite(loss):                       # skip batches with no response tokens
            (loss / ACCUM).backward()
        if micro % ACCUM == 0:
            lr = get_lr(update, WARMUP, total_updates, PEAK_LR, MIN_LR)
            for g in optimizer.param_groups:
                g["lr"] = lr
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            update += 1
            if update % 200 == 0:
                tr, va = evaluate_model(model, train_loader, val_loader, device, 50)
                tqdm.write(f"update {update}/{total_updates} | lr {lr:.2e} | train {tr:.3f} | val {va:.3f} | {(time.time()-t0)/60:.0f} min")
                with open("sft_v2_losses.csv", "a") as f:
                    f.write(f"{update},{tr},{va}\n")
    torch.save(model.state_dict(), f"my-finetuned-gpt-v2-epoch{epoch + 1}.pth")   # one checkpoint per epoch
    print(f"epoch {epoch + 1} done, saved my-finetuned-gpt-v2-epoch{epoch + 1}.pth")

torch.save(model.state_dict(), "my-finetuned-gpt-v2.pth")
print(f"Training completed in {(time.time() - t0) / 60:.1f} minutes.")

epoch 1/3:   0%|          | 0/25000 [00:00<?, ?it/s]

update 200/9375 | lr 2.00e-05 | train 2.245 | val 2.271 | 98 min
update 400/9375 | lr 2.00e-05 | train 2.113 | val 2.196 | 164 min
update 600/9375 | lr 1.99e-05 | train 2.086 | val 2.151 | 170 min
update 800/9375 | lr 1.97e-05 | train 2.085 | val 2.119 | 176 min
update 1000/9375 | lr 1.96e-05 | train 2.053 | val 2.091 | 183 min
update 1200/9375 | lr 1.94e-05 | train 2.092 | val 2.068 | 189 min
update 1400/9375 | lr 1.91e-05 | train 2.008 | val 2.049 | 196 min
update 1600/9375 | lr 1.89e-05 | train 1.984 | val 2.033 | 202 min
update 1800/9375 | lr 1.86e-05 | train 1.945 | val 2.020 | 206 min
update 2000/9375 | lr 1.82e-05 | train 1.940 | val 2.007 | 209 min
update 2200/9375 | lr 1.78e-05 | train 1.896 | val 1.995 | 211 min
update 2400/9375 | lr 1.74e-05 | train 1.852 | val 1.983 | 214 min
update 2600/9375 | lr 1.70e-05 | train 2.039 | val 1.975 | 217 min
update 2800/9375 | lr 1.65e-05 | train 1.884 | val 1.967 | 219 min
update 3000/9375 | lr 1.60e-05 | train 1.918 | val 1.959 | 222 min


epoch 2/3:   0%|          | 0/25000 [00:00<?, ?it/s]

update 3200/9375 | lr 1.55e-05 | train 1.863 | val 1.949 | 225 min
update 3400/9375 | lr 1.49e-05 | train 1.864 | val 1.943 | 227 min
update 3600/9375 | lr 1.44e-05 | train 1.865 | val 1.938 | 230 min
update 3800/9375 | lr 1.38e-05 | train 1.925 | val 1.934 | 233 min
update 4000/9375 | lr 1.32e-05 | train 1.842 | val 1.926 | 236 min
update 4200/9375 | lr 1.26e-05 | train 1.789 | val 1.922 | 238 min
update 4400/9375 | lr 1.20e-05 | train 1.823 | val 1.919 | 241 min
update 4600/9375 | lr 1.14e-05 | train 1.814 | val 1.916 | 244 min
update 4800/9375 | lr 1.08e-05 | train 1.879 | val 1.910 | 246 min
update 5000/9375 | lr 1.02e-05 | train 1.838 | val 1.906 | 249 min
update 5200/9375 | lr 9.60e-06 | train 1.803 | val 1.902 | 252 min
update 5400/9375 | lr 9.00e-06 | train 1.830 | val 1.899 | 254 min
update 5600/9375 | lr 8.41e-06 | train 1.920 | val 1.896 | 257 min
update 5800/9375 | lr 7.83e-06 | train 1.781 | val 1.893 | 260 min
update 6000/9375 | lr 7.27e-06 | train 1.752 | val 1.891 | 263

epoch 3/3:   0%|          | 0/25000 [00:00<?, ?it/s]

update 6400/9375 | lr 6.20e-06 | train 1.807 | val 1.886 | 268 min
update 6600/9375 | lr 5.69e-06 | train 1.773 | val 1.885 | 271 min
update 6800/9375 | lr 5.21e-06 | train 1.824 | val 1.883 | 273 min
update 7000/9375 | lr 4.76e-06 | train 1.820 | val 1.882 | 276 min
update 7200/9375 | lr 4.34e-06 | train 1.768 | val 1.879 | 279 min
update 7400/9375 | lr 3.94e-06 | train 1.816 | val 1.878 | 281 min
update 7600/9375 | lr 3.58e-06 | train 1.834 | val 1.878 | 284 min
update 7800/9375 | lr 3.25e-06 | train 1.787 | val 1.877 | 287 min
update 8000/9375 | lr 2.96e-06 | train 1.749 | val 1.875 | 289 min
update 8200/9375 | lr 2.70e-06 | train 1.785 | val 1.875 | 292 min
update 8400/9375 | lr 2.49e-06 | train 1.762 | val 1.873 | 295 min
update 8600/9375 | lr 2.31e-06 | train 1.788 | val 1.873 | 297 min
update 8800/9375 | lr 2.17e-06 | train 1.802 | val 1.872 | 300 min
update 9000/9375 | lr 2.07e-06 | train 1.724 | val 1.872 | 303 min
update 9200/9375 | lr 2.02e-06 | train 1.758 | val 1.871 | 305

In [27]:
# Save the final fine-tuned model
import torch
import os

save_path = "my-finetuned-gpt-v2.pth"

torch.save(model.state_dict(), save_path)

print(f"Model saved successfully: {os.path.abspath(save_path)}")
print(f"File size: {os.path.getsize(save_path) / (1024**2):.2f} MB")

Model saved successfully: c:\Users\SAI SIDDARDHA\Documents\programming this pc\built a llm from scrach book\my-finetuned-gpt-v2.pth
File size: 669.90 MB


In [25]:
import sys, gc, torch
sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect(); torch.cuda.empty_cache()
print("allocated GB:", round(torch.cuda.memory_allocated()/1e9, 2))
print("reserved  GB:", round(torch.cuda.memory_reserved()/1e9, 2))

allocated GB: 2.79
reserved  GB: 3.62


In [26]:
import gc, torch
for name in ["ckpt", "checkpoint"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()
print("allocated GB:", round(torch.cuda.memory_allocated()/1e9, 2))
print("reserved  GB:", round(torch.cuda.memory_reserved()/1e9, 2))

allocated GB: 2.09
reserved  GB: 2.89


In [28]:
import gradio as gr
import torch
import gc
from pathlib import Path

# ---------- Load the FINAL fine-tuned model ----------
chat_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = Path("my-finetuned-gpt-v2.pth")

# Fallback to the final epoch checkpoint if needed
if not checkpoint_path.exists():
    checkpoint_path = Path("my-finetuned-gpt-v2-epoch3.pth")

if not checkpoint_path.exists():
    raise FileNotFoundError(
        "Could not find my-finetuned-gpt-v2.pth "
        "or my-finetuned-gpt-v2-epoch3.pth"
    )

# Reuse the existing model to avoid allocating a second model on the GPU
if "model" in globals() and isinstance(model, GPTModel):
    chat_model = model
else:
    chat_model = GPTModel(GPT_CONFIG_163M)

state_dict = torch.load(
    checkpoint_path, map_location="cpu", weights_only=True
)
chat_model.load_state_dict(state_dict)
del state_dict

chat_model.to(chat_device).eval()

# Release optimizer state memory; training is already complete
if "optimizer" in globals():
    del optimizer

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Loaded {checkpoint_path} on {chat_device}")


# ---------- Raw prompt generation ----------
def generate_raw(
    prompt,
    max_new_tokens=100,
    temperature=0.7,
    top_k=40,
    repetition_penalty=1.2,
):
    if not prompt or not prompt.strip():
        return "Enter a prompt."

    # IMPORTANT: tokenize the exact prompt as typed.
    # No instruction/input/response formatting is added.
    ids = text_to_token_ids(prompt, tokenizer).to(chat_device)
    prompt_len = ids.shape[1]
    context_size = GPT_CONFIG_163M["context_length"]

    with torch.inference_mode():
        for _ in range(int(max_new_tokens)):
            logits = chat_model(ids[:, -context_size:])[:, -1, :].float()

            # Penalize previously generated tokens
            if repetition_penalty > 1.0:
                for token_id in set(ids[0, prompt_len:].tolist()):
                    score = logits[0, token_id]
                    logits[0, token_id] = (
                        score * repetition_penalty
                        if score < 0
                        else score / repetition_penalty
                    )

            # Optional top-k filtering
            if top_k and int(top_k) > 0:
                k = min(int(top_k), logits.shape[-1])
                top_values, _ = torch.topk(logits, k, dim=-1)
                threshold = top_values[:, -1].unsqueeze(-1)
                logits = torch.where(
                    logits < threshold,
                    torch.full_like(logits, float("-inf")),
                    logits,
                )

            # Greedy decoding when temperature is zero;
            # sampling otherwise
            if temperature <= 0:
                next_token = torch.argmax(
                    logits, dim=-1, keepdim=True
                )
            else:
                probs = torch.softmax(logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

            # Stop at GPT-2's end-of-text token
            if next_token.item() == 50256:
                break

            ids = torch.cat((ids, next_token), dim=1)

    # Return only the newly generated continuation
    generated_ids = ids[0, prompt_len:].cpu().tolist()
    return tokenizer.decode(generated_ids).strip()


# ---------- Local Gradio interface ----------
with gr.Blocks(title="My Fine-Tuned GPT — Raw Prompt Test") as demo:
    gr.Markdown("# My Fine-Tuned GPT")
    gr.Markdown(
        "Enter any raw prompt. It is passed directly to the model "
        "without the instruction-training template."
    )

    prompt_box = gr.Textbox(
        label="Prompt",
        placeholder="What is the capital of France?",
        lines=4,
    )

    with gr.Row():
        max_tokens = gr.Slider(
            minimum=1, maximum=256, value=100, step=1,
            label="Max new tokens",
        )
        temperature = gr.Slider(
            minimum=0, maximum=1.5, value=0.7, step=0.05,
            label="Temperature (0 = greedy)",
        )

    with gr.Row():
        top_k = gr.Slider(
            minimum=0, maximum=100, value=40, step=1,
            label="Top-k (0 = disabled)",
        )
        rep_penalty = gr.Slider(
            minimum=1.0, maximum=2.0, value=1.2, step=0.05,
            label="Repetition penalty",
        )

    generate_button = gr.Button("Generate", variant="primary")

    output_box = gr.Textbox(
        label="Generated continuation",
        lines=10,
    )

    generate_button.click(
        fn=generate_raw,
        inputs=[prompt_box, max_tokens, temperature, top_k, rep_penalty],
        outputs=output_box,
    )

    prompt_box.submit(
        fn=generate_raw,
        inputs=[prompt_box, max_tokens, temperature, top_k, rep_penalty],
        outputs=output_box,
    )

demo.launch(server_name="127.0.0.1", share=False)

Loaded my-finetuned-gpt-v2.pth on cuda
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
